# 🔬 Replikasi Eksperimen 3: Repeated LOPO-CV & Analisis Kemampuan Dataset IDoFT
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Dataset Acuan**: IDoFT (Illinois Dataset of Flaky Tests) Java Projects (`pr-data.csv` & `gr-data.csv`) aligned with FlakeFlagger 23 Features  
**Rujukan Paper**: Afeltra et al. (IEEE Access 2024) — *A Large-Scale Empirical Investigation Into Cross-Project Flaky Test Prediction*  
**Hardware Target**: NVIDIA GeForce RTX 4050 Laptop GPU (CUDA hist) & CPU Multi-threading  
**Metodologi**: Repeated Leave-One-Project-Out Cross-Validation (5 Repetisi x 24 Pipelines x 17 Folds = **2,040 Model Folds**).

---

## 📋 Ikhtisar & Latar Belakang Replikasi
Eksperimen ini mereplikasi metodologi pengujian berulang (*repeated LOPO-CV*) dan analisis kemampuan dataset individual dari **Eksperimen 3**, namun menggunakan **Dataset IDoFT** (sebagaimana digunakan oleh paper Afeltra et al., 2024).

Dataset mencakup **17 proyek Java** (12.626 baris pengujian, 264 flaky test ground truth riil, tingkat ketimpangan ekstrem 2,09%).
Sebanyak 24 konfigurasi pipeline (3 Feature Scaling x 4 Imbalance Techniques x 2 Machine Learning Classifiers) diuji melintasi 5 *random seed* independen ([42, 100, 2024, 777, 999]).


## 1. Verifikasi GPU NVIDIA RTX 4050 & Lingkungan ML


In [ ]:
import sys
from pathlib import Path

src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.append(src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()


## 2. Ingestion & Verifikasi Dataset IDoFT (17 Proyek, 12.626 Baris)


In [ ]:
from data_loader import load_idoft_dataset
df, features, projects = load_idoft_dataset()

print(f"Total Baris: {len(df):,} | Proyek: {len(projects)} | Total Fitur Prediktif: {len(features)}")
print(f"Flaky Tests IDoFT: {df['flaky'].sum():,} ({df['flaky'].mean()*100:.2f}%)")
print("Daftar Proyek IDoFT:", projects)


## 3. Eksekusi Repeated LOPO-CV (24 Pipelines x 5 Seeds x 17 Folds = 2,040 Model Evaluasi)


In [ ]:
from repeated_lopo_runner import run_repeated_lopo_experiments

# Menjalankan evaluasi 2.040 folds dengan GPU Acceleration
results = run_repeated_lopo_experiments(use_gpu=True)


## 4. Analisis Konsistensi & Stabilitas Pipeline Lintas Seeds


In [ ]:
from consistency_analyzer import analyze_pipeline_consistency
import pandas as pd

df_cons = analyze_pipeline_consistency()
display_cols = [
    'Pipeline_ID', 'Scaling', 'Imbalance', 'Model',
    'Mean_F1_Mean', 'Mean_F1_Active_13_Mean', 'Mean_F1_Std', 'Mean_F1_CV_%',
    'Mean_Recall_Mean', 'Mean_Precision_Mean', 'Mean_PR_AUC_Mean'
]
df_cons[display_cols].head(10)


## 5. Profiling Kemampuan 17 Dataset Proyek & Root Cause Analysis


In [ ]:
from dataset_profiler import profile_all_datasets

df_profile, analysis_dict = profile_all_datasets()
df_profile[[
    'Target_Project', 'Flaky_Count', 'Test_Size', 'Flaky_Rate_%',
    'Overall_Mean_F1', 'Overall_Std_F1', 'Best_Pipeline', 'Best_Pipeline_F1', 'Tier_Code'
]].head(17)


## 6. Uji Signifikansi Statistik (Friedman & Nemenyi Post-Hoc)


In [ ]:
from statistical_tests import run_repeated_statistical_analysis

df_stat_f1 = run_repeated_statistical_analysis(metric_col="F1")
df_stat_prauc = run_repeated_statistical_analysis(metric_col="PR_AUC")
print("Top 5 Peringkat Friedman Nemenyi (F1):")
print(df_stat_f1.head(5).to_string(index=False))


## 7. Pembuatan Visualisasi Publikasi Ilmiah 300 DPI


In [ ]:
from visualizer import plot_all_experiment_3_figures
plot_all_experiment_3_figures()
